# iot-sae-ids — Colab runner

Runs the IoT sparse-autoencoder IDS experiments on Colab Pro (T4 GPU).

**Before the first run**, put on Google Drive under `WORK_DIR` (default `MyDrive/iot-sae-ids-work`):

```
iot-sae-ids-work/
├── iot-sae-ids.bundle          # only if REPO_SOURCE = "bundle" (see README → Colab)
└── data/raw/
    ├── ton_iot/Train_Test_Network.csv
    ├── unsw_nb15/UNSW_NB15_training-set.csv
    ├── unsw_nb15/UNSW_NB15_testing-set.csv
    └── rt_iot2022/RT_IOT2022.csv      # optional: fetched from UCI if missing
```

Data, cached features and **all results live on Drive** (`IDS_DATA_ROOT`, `IDS_RESULTS_ROOT`), so a
disconnect loses nothing: rerun the setup cells, then the run cell again; finished runs are skipped.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# --- Settings -------------------------------------------------------------------
WORK_DIR    = "/content/drive/MyDrive/iot-sae-ids-work"
REPO_SOURCE = "bundle"   # "bundle" (git bundle on Drive) or "github"
REPO_URL    = ""         # e.g. "https://github.com/<you>/iot-sae-ids.git" when REPO_SOURCE="github"
BRANCH      = "main"
CODE_DIR    = "/content/iot-sae-ids"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
pathlib.Path(WORK_DIR, "data", "raw").mkdir(parents=True, exist_ok=True)
pathlib.Path(WORK_DIR, "results").mkdir(parents=True, exist_ok=True)

## 1. Get the code

In [ ]:
import os, subprocess
src = os.path.join(WORK_DIR, "iot-sae-ids.bundle") if REPO_SOURCE == "bundle" else REPO_URL
if not os.path.isdir(os.path.join(CODE_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, src, CODE_DIR], check=True)
else:  # refresh to the latest bundle / remote commit
    subprocess.run(["git", "-C", CODE_DIR, "fetch", src, f"{BRANCH}:refs/remotes/src/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "reset", "--hard", f"src/{BRANCH}"], check=True)
!git -C {CODE_DIR} log --oneline -3

## 2. Install and point paths at Drive

In [ ]:
%cd {CODE_DIR}
!pip install -q -r requirements-tf.txt && pip install -q -e .
os.environ["IDS_DATA_ROOT"] = os.path.join(WORK_DIR, "data")
os.environ["IDS_RESULTS_ROOT"] = os.path.join(WORK_DIR, "results")
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Tests (TensorFlow tests run here)

In [ ]:
!python -m pytest -q

## 4. Smoke test — all 12 combinations on a 2% sample (CHECKPOINT 2)
Writes `results/tables/smoke_test.md` on Drive.

In [ ]:
!python scripts/smoke_test.py --sample 0.02